# 03 - Feature Engineering, Leakage Detection & Splitting
In this comprehensive notebook, we engineer EVERY feature required by the project specifications, rigorously check for data leakage, and chronologically split our data.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

cleaned_data_path = '../../../data/processed/attendance_cleaned.csv'
df = pd.read_csv(cleaned_data_path)

# Ensure Date is properly formatted
df['Date'] = pd.to_datetime(df['Date'])
df['Start_Time_DT'] = pd.to_datetime(df['Start_Time'], errors='coerce').dt.time

# ⚠️ CRITICAL: Must sort chronologically before any feature engineering to prevent leakage!
df.sort_values(by=['Date', 'Start_Time_DT'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,Date,Day_of_Week,Start_Time,End_Time,Subject,Faculty_ID,Semester,Branch,Section,Classroom,Total_Enrolled,Students_Present,Previous_Lecture_Attendance,Practical_Theory,Internal_Test_Week,Holiday_Before_After,Weather,Special_Event,Start_Time_DT,Attendance_Percentage
0,2026-04-10,Friday,08:30:00,09:15:00,Mobile Application Development,SSP,3,MCA,A&B,Computer Lab,204,62,30,Theory,No,Before,Rainy,No,08:30:00,30.39
1,2026-04-10,Friday,09:15:00,10:15:00,Mobile Application Development,SSP,3,MCA,A&B,Computer Lab,204,60,62,Theory,No,Before,Rainy,Yes,09:15:00,29.41
2,2026-04-10,Friday,10:15:00,11:15:00,Data Science & Machine Learning,SP_MLK,3,MCA,A&B,Computer Lab,204,33,14,Theory,No,Before,Sunny,No,10:15:00,16.18
3,2026-04-10,Friday,11:15:00,12:15:00,Principles of Cloud Management and Security,KSD_SDM,3,MCA,A&B,Computer Lab,204,21,47,Theory,No,Before,Sunny,Yes,11:15:00,10.29
4,2026-04-13,Monday,08:30:00,09:15:00,Mobile Application Development,SSP,3,MCA,A&B,Computer Lab,204,39,60,Theory,No,Before,Cloudy,No,08:30:00,19.12


### Step 1: Global Temporal Features
Features related to the passage of time across the entire semester.

In [2]:
# 1. Day of Semester (0 on the first recorded day, increasing sequentially)
semester_start = df['Date'].min()
df['Day_of_Semester'] = (df['Date'] - semester_start).dt.days

# 2. Week Number (Standard ISO Calendar Week)
df['Week_Number'] = df['Date'].dt.isocalendar().week.astype(int)

# 3. Time-of-Day Clusters (Morning vs Afternoon)
df['Time_of_Day_Cluster'] = df['Start_Time_DT'].apply(lambda x: 'Morning' if x.hour < 12 else 'Afternoon')

# 3.5 Lunch Break Flag (Classes starting between 1:00 PM and 2:59 PM)
df['Is_Post_Lunch_Class'] = df['Start_Time_DT'].apply(lambda x: 1 if 13 <= x.hour <= 14 else 0)

df[['Date', 'Start_Time', 'Day_of_Semester', 'Week_Number', 'Time_of_Day_Cluster', 'Is_Post_Lunch_Class']].head()

,Date,Start_Time,Day_of_Semester,Week_Number,Time_of_Day_Cluster,Is_Post_Lunch_Class
0,2026-04-10,08:30:00,0,15,Morning,0
1,2026-04-10,09:15:00,0,15,Morning,0
2,2026-04-10,10:15:00,0,15,Morning,0
3,2026-04-10,11:15:00,0,15,Morning,0
4,2026-04-13,08:30:00,3,16,Morning,0


### Step 2: Subject-Specific Historical Tracking (Using `.shift()`)
We calculate historical features *per subject and section* using only past data.

In [3]:
group = df.groupby(['Subject', 'Section'])
global_mean_attendance = df['Attendance_Percentage'].mean()

# 4. Previous Lecture Attendance (Chronological Daily Momentum)
df['Previous_Lecture_Attendance_Pct'] = df.groupby('Date')['Attendance_Percentage'].shift(1)
df['Previous_Lecture_Attendance_Pct'] = df['Previous_Lecture_Attendance_Pct'].fillna(0)

# 5. Days Since Previous Lecture (Gap)
df['Prev_Date'] = group['Date'].shift(1)
df['Gap_Since_Previous_Lecture_Days'] = (df['Date'] - df['Prev_Date']).dt.days
df['Gap_Since_Previous_Lecture_Days'] = df['Gap_Since_Previous_Lecture_Days'].fillna(0).astype(int)
df.drop(columns=['Prev_Date'], inplace=True)

# 6. Rolling Average (Last 3 Lectures)
df['Rolling_Avg_3_Lectures'] = group['Attendance_Percentage'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df['Rolling_Avg_3_Lectures'] = df['Rolling_Avg_3_Lectures'].fillna(0)

# 7. Consecutive Lecture Count (Streak of classes with <= 2 days gap)
df['Is_Consecutive'] = (df['Gap_Since_Previous_Lecture_Days'] <= 2).astype(int)
df['Consecutive_Lecture_Count'] = group['Is_Consecutive'].cumsum()

# 8. Monthly Expanding Average
df['Month'] = df['Date'].dt.month
df['Monthly_Expanding_Mean'] = df.groupby(['Month', 'Subject', 'Section'])['Attendance_Percentage'].transform(lambda x: x.shift(1).expanding().mean())
df['Monthly_Expanding_Mean'] = df['Monthly_Expanding_Mean'].fillna(0)

### Step 3: Event Flags & Categorical Encoding
Converting textual disruptions into machine-readable numeric flags, and one-hot encoding categories.

In [4]:
# 9. Holiday Adjacent Flag
df['Is_Holiday_Adjacent'] = df['Holiday_Before_After'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# 10. Week-Before-Exam Flag
df['Week_Before_Exam_Flag'] = df['Internal_Test_Week'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# 11. One-Hot Encoding for remaining categorical string variables
cols_to_encode = ['Day_of_Week', 'Subject', 'Weather', 'Practical_Theory', 'Time_of_Day_Cluster']
df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

print(f"Final Engineered Dataset Shape: {df_encoded.shape}")

Final Engineered Dataset Shape: (515, 56)


### Step 4: Leakage Detection Gate
Mathematically proving that our engineered historical features do not accidentally contain the future `Attendance_Percentage`.

In [5]:
engineered_numeric_cols = [
    'Previous_Lecture_Attendance_Pct', 'Gap_Since_Previous_Lecture_Days', 
    'Rolling_Avg_3_Lectures', 'Consecutive_Lecture_Count', 'Monthly_Expanding_Mean',
    'Day_of_Semester', 'Is_Post_Lunch_Class'
]

correlation_matrix = df_encoded[['Attendance_Percentage'] + engineered_numeric_cols].corr()
print("Correlation with Target (Attendance_Percentage):")
print(correlation_matrix['Attendance_Percentage'])

for col in engineered_numeric_cols:
    corr = correlation_matrix.loc[col, 'Attendance_Percentage']
    assert corr < 0.95, f"LEAKAGE DETECTED! {col} is perfectly correlated with the target."

print("\n✅ LEAKAGE DETECTION PASSED: No engineered feature is illegally predicting the target.")

Correlation with Target (Attendance_Percentage):
Attendance_Percentage              1.000000
Previous_Lecture_Attendance_Pct    0.309754
Gap_Since_Previous_Lecture_Days   -0.052458
Rolling_Avg_3_Lectures             0.238154
Consecutive_Lecture_Count          0.130449
Monthly_Expanding_Mean             0.137253
Day_of_Semester                   -0.078738
Is_Post_Lunch_Class               -0.069368
Name: Attendance_Percentage, dtype: float64

✅ LEAKAGE DETECTION PASSED: No engineered feature is illegally predicting the target.


### Step 5: Temporal Splitting (Chronological)
Splitting into Train (70%), Validation (15%), and Test (15%) strictly based on time.

In [6]:
from sklearn.model_selection import train_test_split

drop_cols = ['Start_Time', 'End_Time', 'Faculty_ID', 'Branch', 'Section', 'Classroom', 'Holiday_Before_After', 'Internal_Test_Week', 'Special_Event', 'Start_Time_DT', 'Previous_Lecture_Attendance', 'Gap_Since_Previous_Lecture', 'Assignment_Due', 'Attendance_Category', 'Lecture_No']
df_final = df_encoded.drop(columns=[col for col in drop_cols if col in df_encoded.columns])

# 70% Train, 30% Temp (Val + Test)
train_df, temp_df = train_test_split(df_final, test_size=0.30, shuffle=False)

# 15% Val, 15% Test
val_df, test_df = train_test_split(temp_df, test_size=0.50, shuffle=False)

print(f"Train set: {train_df.shape}")
print(f"Validation set: {val_df.shape}")
print(f"Test set: {test_df.shape}")

assert train_df['Date'].max() <= val_df['Date'].min(), "Temporal Leakage: Train data overlaps Validation data!"
assert val_df['Date'].max() <= test_df['Date'].min(), "Temporal Leakage: Validation data overlaps Test data!"
print("\n✅ TEMPORAL SPLITS VERIFIED: Data is strictly chronological. No future peeking.")

train_df = train_df.drop(columns=['Date'])
val_df = val_df.drop(columns=['Date'])
test_df = test_df.drop(columns=['Date'])

train_df.to_csv('../../../data/processed/train.csv', index=False)
val_df.to_csv('../../../data/processed/val.csv', index=False)
test_df.to_csv('../../../data/processed/test.csv', index=False)

Train set: (360, 45)
Validation set: (77, 45)
Test set: (78, 45)

✅ TEMPORAL SPLITS VERIFIED: Data is strictly chronological. No future peeking.
